Below is a complete inventory of every controller in the project that uses discrete‑time CBFs, how the constraints are enforced, and which pedestrians are taken into account.

---

### Controllers that employ DCBFs

1. **DCBF‑MPPI** – `dcbf_mppi.py`  
2. **DCBF‑NMPC** – `dcbf_nmpc.py`  
3. **DCBF‑MPCC‑MPPI** – `dcbf_mpcc_mppi.py` (inherits DCBF‑MPPI behaviour)

Other controllers (MPPI, RiskAwareMPPI, NMPC, StandardMPC, etc.) do **not** use CBFs; they rely on soft obstacle penalties or implicit sampling‑based avoidance.

---

### How each controller uses the DCBF

#### DCBF‑MPPI and DCBF‑MPCC‑MPPI

| Property | Implementation |
|----------|----------------|
| **When is DCBF applied?** | After the nominal MPPI command is computed (including maneuvers, DOB, LPFs). It is a **post‑hoc safety filter**. |
| **Type of constraint** | **Hard constraint** on the executed command. The projection attempts to find a `(v, ω)` that strictly satisfies the discrete‑time CBF condition for all obstacles. |
| **How is a violation handled?** | The nominal command is checked with `_cbf_value_for_state`. If `min_dcbf < 0`, the command is replaced by the output of the chosen projection method (grid search, scipy optimisation, or analytical). All projection methods are designed to produce a command with `min_dcbf ≥ 0`, but if none exist they fall back to `v=0, ω=0` – so the robot stops rather than collide. |
| **Is slack used?** | Only in the optimisation‑based projection; a slack variable is added to the constraints and heavily penalised in the cost, but the final projected command still attempts to satisfy DCBF. The grid search and analytical methods do not use slack; they directly search for a safe velocity. |
| **Obstacle set** | **All obstacles** passed to the controller. In `robot_demo.py` with safety mode `"all"`, this includes every pedestrian, the user, static and dynamic obstacles. Pedestrians beyond the immediate vicinity are still included, but the projection functions process obstacles sequentially by distance, so they effectively handle all. |

#### DCBF‑NMPC

| Property | Implementation |
|----------|----------------|
| **When is DCBF applied?** | Inside the NMPC optimisation itself, as **explicit inequality constraints** for each prediction step. |
| **Type of constraint** | **Hard constraint with slack**. The DCBF condition `h_next - (1-γ)·h_curr + slack ≥ 0` is added to the CasADi NLP. The slack variable `δ ≥ 0` is shared across all obstacles. |
| **How is a violation handled?** | The slack allows the constraint to be temporarily violated during the optimisation if no feasible solution exists. However, a large quadratic penalty `ρ·δ²` (ρ = 10⁶) is added to the cost, so the solver only uses slack when absolutely necessary. If the NLP solver fails entirely, a fallback projection (grid/optimisation/analytical) is applied, guaranteeing a safe command. |
| **Is slack used?** | Yes – it is the core mechanism that keeps the NLP feasible at all times. The penalty ensures the robot respects the DCBF whenever physically possible. |
| **Obstacle set** | **All obstacles** are encoded as DCBF constraints. The CasADi formulation includes every circle and wall for every prediction step. There is no vicinity filter; the NLP solver handles the full set. |

---

### Summary table

| Controller | CBF location | Hard/Soft | Slack? | Obstacle coverage |
|------------|--------------|-----------|--------|-------------------|
| DCBF‑MPPI | Post‑hoc filter | Hard | Only in opt. projection | All obstacles |
| DCBF‑MPCC‑MPPI | Post‑hoc filter (inherited) | Hard | Same as above | All obstacles |
| DCBF‑NMPC | Inside NLP constraints | Hard with slack | Yes – single slack variable | All obstacles |

---

### Key points

- **No soft CBFs** exist in the project. Every CBF implementation aims to strictly enforce the discrete‑time safety condition. When a strict solution is impossible, the slack variable (NMPC) or the fallback to zero velocity (MPPI variants) provides a graceful degradation.
- **All pedestrians are considered**, not just nearby ones. In the MPPI variants, the proximity of obstacles determines the cost / violation magnitude, but every pedestrian’s CBF condition is evaluated. In NMPC, the NLP explicitly includes every obstacle.
- The CBF conditions are always evaluated in discrete time: `h(x_{k+1}) ≥ (1-γ) h(x_k)`. The forward dynamics for the robot are either the Euler approximation (in the MPPI filters) or the exact unicycle kinematics (in the NMPC solver).
---
---
---

### 1 – Restrict DCBFs to a 4.0 m vicinity

We only need to filter the `circles` array (obstacles) before they enter the CBF checks. Walls are few and can be kept as‑is, but you may also filter them for consistency.

Add this helper inside each controller file that uses DCBFs:

```python
def _filter_by_distance(circles, robot_x, robot_y, radius=4.0):
    if circles.shape[0] == 0:
        return circles
    dist = np.hypot(circles[:, 0] - robot_x, circles[:, 1] - robot_y)
    return circles[dist <= radius]
```

---

#### a) DCBF‑MPPI & DCBF‑MPCC‑MPPI (post‑hoc filter)

In `DCBFMPPIController.compute_velocity`, right after converting obstacles to `circles, walls`, insert:

```python
        circles = _filter_by_distance(circles, x, y, radius=4.0)
```

Now the safety check and all projection methods see only obstacles within 4 m.

---

#### b) DCBF‑NMPC (constraints inside NLP)

In `DCBFNMPCController._solve_mpc`, after the CasADi problem is created but before building DCBF constraints, filter the numpy arrays:

```python
        # ---- Filter obstacles to vicinity ----
        circles = _filter_by_distance(circles, x0[0], x0[1], radius=4.0)
        walls  = _filter_by_distance_walls(walls, x0[0], x0[1], radius=4.0)   # optional
```

The `walls` filtering can be done analogously (distance to line segment). Then construct the CasADi constraints as usual – they will only involve the nearby obstacles.

---

### 2 – Proposal to merge all DCBF constraints into a single condition

For a collection of obstacles $i=1,\dots,N$, the individual discrete‑time CBF conditions are:

$$
h_i(x_{k+1}) \;\ge\; (1-\gamma)\,h_i(x_k)
$$

To enforce them **simultaneously**, we can define a **global barrier function**:

$$
H(x) \;=\; \min_i\; h_i(x)
$$

Because if $H(x_{k+1}) \ge (1-\gamma) H(x_k)$ and $H(x_k)\ge 0$, then all $h_i$ stay non‑negative. This single inequality is **sufficient** for safety.

#### Why it’s not always practical

- $\min$ is non‑smooth – gradient‑based solvers (IPOPT) may struggle at points where the minimum is attained by two obstacles.
- It is **conservative** – the worst‑case obstacle dictates the allowed motion, even if other obstacles are far away.

#### Smoother alternatives

1. **Log‑sum‑exp soft‑min**:
   $$
   H_\beta(x) = -\frac{1}{\beta}\ln\sum_i e^{-\beta h_i(x)}
   $$
   As $\beta\to\infty$ it approaches the minimum. Smooth, differentiable, and works inside CasADi.

2. **Weighted product** (not true merging):
   $$
   \prod_i \mathbb{1}[h_i(\cdot)\ge 0]
   $$
   is binary – not useful for continuous optimisation.

3. **Single slack for all** (already used in DCBF‑NMPC):
   Keep individual constraints but share a single slack variable $\delta$:
   $$
   h_i(x_{k+1}) \ge (1-\gamma)h_i(x_k) - \delta, \quad \delta\ge 0
   $$
   and heavily penalise $\delta^2$. This is simple, differentiable, and already implemented.

#### Recommendation

For **DCBF‑MPPI** (post‑hoc filter), the current per‑obstacle check is fast and robust – merging wouldn’t add value.  
For **DCBF‑NMPC**, the shared‑slack approach already handles multiple constraints efficiently. If you ever want a single constraint for speed, replace the per‑obstacle loop with the **log‑sum‑exp soft‑min** of the CBF residues. This would produce one smooth inequality per time step, reducing the NLP size significantly, at the cost of some conservativeness and tuning of $\beta$.

We’ll replace the per‑obstacle DCBF constraints with a single smooth inequality using the **log‑sum‑exp soft‑minimum**. This drastically reduces the number of constraints – for DCBF‑NMPC it cuts the NLP size significantly and speeds up IPOPT; for the optimisation‑based projection in DCBF‑MPPI it also simplifies the QP.

---

### 1. Mathematical formulation

For each obstacle $i$ we have the discrete‑time CBF residual:

$$
r_i = h_i(x_{k+1}) - (1-\gamma)\,h_i(x_k)
$$

We need $\min_i r_i \ge 0$ (with slack if needed).  
The **smooth approximation** of the minimum is the log‑sum‑exp:

$$
R_\beta = -\frac{1}{\beta}\,\ln\sum_i \exp\!\big(-\beta\, r_i\big)
$$

$R_\beta \le \min_i r_i$ always, and as $\beta\to\infty$, $R_\beta \to \min_i r_i$.  
A single inequality  

$$
R_\beta + \delta \ge 0
$$

replaces all the individual $r_i + \delta \ge 0$ constraints. $\beta$ controls the sharpness; start with $\beta = 10$.

---

### 2. Changes in DCBF‑NMPC

**File:** `dcbf_nmpc.py` (the NLP solver part)

Currently you loop over obstacles and add constraints like:

```python
        for i in range(circles.shape[0]):
            ...
            opti.subject_to(h_next - (1 - self.gamma)*h_k + slack >= 0)
```

Replace the **entire obstacle loop** (both circles and walls) with:

```python
        # ---- Unified DCBF via log‑sum‑exp ----
        beta = 10.0   # tunable sharpness
        for k in range(H):
            # collect residuals for all obstacles at prediction step k
            residuals = []
            # circles
            for i in range(circles.shape[0]):
                cx, cy, cr = circles[i]
                r_total = safe_dist + cr
                h_k = ca.sumsqr(X[:2,k] - ca.vertcat(cx,cy)) - r_total**2
                h_next = ca.sumsqr(X[:2,k+1] - ca.vertcat(cx,cy)) - r_total**2
                r_i = h_next - (1.0 - self.gamma)*h_k
                residuals.append(r_i)
            # walls (approximate with point distance, as before)
            for j in range(walls.shape[0]):
                # ... compute h_k, h_next for wall j ...
                r_i = h_next - (1.0 - self.gamma)*h_k
                residuals.append(r_i)

            # log‑sum‑exp of negative residuals
            if len(residuals) == 0:
                continue
            res_stack = ca.vertcat(*residuals)   # vector of residuals
            lse = -(1.0/beta) * ca.log(ca.sum1(ca.exp(-beta * res_stack)))
            opti.subject_to(lse + slack >= 0)
```

Walls can be included in the same `residuals` list; just compute their `h_k`/`h_next` with the closest‑point distance logic.

**Result:** Instead of $N_{\text{obs}} \times H$ constraints, we now have only $H$ constraints. IPOPT will converge much faster.

---

### 3. Changes in the optimisation‑based projection (`dcbf_mppi.py`)

**Function:** `_optimization_projection`

Currently it passes a vector of DCBF constraints to `minimize`.  
Replace the `dcbf_constraints` function and the constraint definition with:

```python
    beta = 10.0
    def unified_constraint(z):
        v, omega, slack = z
        x_next, y_next, _ = diff_drive_dynamics(x, y, theta, v, omega, dt)
        residuals = []
        safe_dist = robot_radius + safety_margin
        for i in range(n_circles):
            cx, cy, r = circles[i, 0], circles[i, 1], circles[i, 2]
            d_total = safe_dist + r
            h_curr = (x - cx)**2 + (y - cy)**2 - d_total**2
            h_next = (x_next - cx)**2 + (y_next - cy)**2 - d_total**2
            residuals.append(h_next - (1.0 - gamma) * h_curr)
        for i in range(n_walls):
            # compute h_curr, h_next for wall i
            residuals.append(h_next - (1.0 - gamma) * h_curr)

        if len(residuals) == 0:
            return np.array([0.0])
        r = np.array(residuals)
        lse = -(1.0/beta) * np.log(np.sum(np.exp(-beta * r)))
        return lse + slack   # must be >= 0

    constraints = {"type": "ineq", "fun": unified_constraint}
```

Now the QP has only **one** inequality constraint instead of $N_{\text{obs}}$.

---

### 4. Tuning $\beta$

- Larger $\beta$ → tighter approximation of the true minimum → safer, but steeper gradients may slow the solver.  
- Start with $\beta = 10$; increase up to 50 if the robot still violates the safety margin.  
- You can make `beta` a class attribute so it’s easy to adjust.

---

### 5. What about the grid search and analytical fallbacks?

The grid search (`cbf_safe_backup_control`) already evaluates all obstacles and picks the safest (v,ω) via a cost function – it’s not constraint‑based, so no change needed.  
The analytical projection processes obstacles sequentially; it also does not need merging.

---

### Summary of steps

1. **Filter obstacles** to a 4 m vicinity (already planned).  
2. **In `dcbf_nmpc.py`**: replace the per‑obstacle DCBF loop with the LSE soft‑min.  
3. **In `dcbf_mppi.py`** (`_optimization_projection`): replace the multiple constraints with a single LSE‑based constraint.  
4. **Tune $\beta$** – start at 10.  
5. **Test** – the robot should behave identically but the NMPC will solve much faster, and the optimisation projection will also be quicker.

---
---
---

The drawn zone is the **1‑sigma contour** of a mood‑driven social potential field. Here’s what it represents and how the robot reacts.

---

### What is being drawn?

For each pedestrian, we compute an **egg‑shaped ellipse** centred at the pedestrian’s position.  
The shape is defined by four semi‑axes:

- **Front (ahead):** $ \sigma_{x,\text{front}} = B_{\text{ped}} \cdot (1 + \lambda) \cdot s $  
- **Back (behind):** $ \sigma_{x,\text{back}} = B_{\text{ped}} \cdot (1 - \lambda) \cdot s $  
- **Left:** $ \sigma_{y,\text{left}} = B_{\text{ped}} \cdot (1 + \frac{\phi}{\pi}) \cdot s $  
- **Right:** $ \sigma_{y,\text{right}} = B_{\text{ped}} \cdot (1 - \frac{\phi}{\pi}) \cdot s $

where  

- $B_{\text{ped}}$ = the calibrated repulsion range (m)  
- $\lambda$ = `lam_base` (anisotropy – makes front larger, back smaller)  
- $\phi$ = `phi_fov` (field of view – widens the side the pedestrian pays more attention to)  
- $s$ = global `social_zone_scale` (tunable, default 2.5)  
- The axes are rotated by $\theta_{\text{ped}} + \theta_{\text{gaze}}$ (heading + gaze offset).

This gives a shape that is **larger in front** (where the pedestrian looks) and **smaller behind**, mimicking how people care more about what they can see.

The contour drawn is the set of points where the Mahalanobis distance  
$ \sqrt{ (dx_b / \sigma_x)^2 + (dy_b / \sigma_y)^2 } = 1 $.  
At this distance the social potential value is $ e^{-0.5} \approx 0.61 $ – it’s the “significant discomfort” boundary.

---

### What does it represent?

It is the **uncomfortable zone** of that pedestrian.  
When the robot’s planned trajectory enters this region, it triggers a social cost that grows as the robot gets closer. The shape is asymmetric because people tolerate closer approaches from the front (they can see you) but dislike being crowded from behind.

---

### How does the robot react?

Inside the MPPI cost function (and similarly for MPCC‑MPPI), at every predicted time step the robot adds a penalty:

$$
\text{cost} \;+\!=\; Q_{\text{social}} \cdot \exp\!\Big(-0.5 \cdot d_{\text{Mah}}^2\Big)
$$

where $d_{\text{Mah}}^2 = (dx_b/\sigma_x)^2 + (dy_b/\sigma_y)^2$.

- $Q_{\text{social}}$ (default 5.0) weights this penalty relative to other objectives (path tracking, speed, etc.).  
- The exponential makes the cost negligible when far away, but significant when inside the contour.  
- Because the cost is **continuously differentiable** (thanks to the ellipse formula), the importance sampling in MPPI naturally steers the robot around the zone without needing hard constraints.

Additionally, if two pedestrians form a group (distance < `r_group`), an extra penalty is applied when the robot is between them, encouraging the robot to go around the whole group rather than cutting through.

---

### Practical effect

- The robot will **plan a path that stays outside the purple egg shapes**.  
- It will give more room to pedestrians who have a larger $B_{\text{ped}}$ (e.g., “Aggressive barger”) and less to calm ones.  
- It will avoid approaching from behind (where the egg is small) and prefer to move into the pedestrian’s field of view if possible, because the cost is lower there (since the zone is wider in front, the gradient pushes the robot sideways into the region of lower cost).  
- The CBF safety layer still provides hard collision guarantees; the social potential is an additional **pro‑active** comfort layer.
---
---
---

I'll give you a thorough, ground-up explanation of **PCA** and the **Silhouette score** – what they are, how they work, why they're used, and how they were applied in your pipeline's Stage 3 clustering.

---

## 1. Principal Component Analysis (PCA)

### What problem does it solve?

Your pipeline extracts many features per window:
- Kinematic: 8 features (speed_mean, speed_std, stop_ratio, accel_mean, accel_max, accel_std, jerk_mean, jerk_max)
- Path: 7 features (tortuosity, mean_curvature, max_curvature, sinuosity, spectral_arc_length, mean_step_angle, step_angle_std)
- Safety: 6 features (min_distance, min_TTC, intrusion_time_frac, num_intrusions, avoidance_deviation, n_neighbors)
- Social: 4 features (fov_occupancy, mutual_attention_count, o_space_ratio, group_affiliation)

Many of these features are **correlated** (e.g. speed_mean and speed_std, or various curvature measures).  
Clustering algorithms like K‑Means and GMM work better when features are:
- Uncorrelated (no redundant information)
- Low‑dimensional (curse of dimensionality – distances become less meaningful in high dimensions)

**PCA** transforms the data into a new coordinate system where the axes are uncorrelated (orthogonal) and ordered by how much variance they capture. You can then keep only the top components that explain most of the variance, reducing noise and redundancy.

### How PCA works – step by step

Suppose you have a data matrix $X$ of shape $(n_{\text{windows}}, d)$ where $d$ is the number of features. After standardisation (mean=0, std=1 for each feature):

1. **Compute the covariance matrix**  
   $\Sigma = \frac{1}{n-1} X^T X$  (a $d \times d$ symmetric matrix).  
   The diagonal entries are the variances of each feature; off-diagonal are covariances.

2. **Find eigenvectors and eigenvalues**  
   Solve $\Sigma \mathbf{v} = \lambda \mathbf{v}$.  
   - Eigenvectors $\mathbf{v}_i$ are the **principal directions** (new axes).  
   - Eigenvalues $\lambda_i$ are the **amount of variance** along that direction.

3. **Sort by eigenvalue descending**  
   The first principal component (PC1) captures the largest variance, PC2 the second largest, etc.

4. **Project the data**  
   Choose the top $k$ eigenvectors (those that cumulatively explain, say, 90% of total variance). Form a matrix $W$ of size $d \times k$. The transformed data is $Z = X W$, now shape $(n_{\text{windows}}, k)$.

**Mathematically**, the total variance of the original data is the sum of all eigenvalues.  
The **explained variance ratio** of PC $i$ is $\frac{\lambda_i}{\sum \lambda_j}$.  
By keeping components that sum to ≥ 0.90, you retain 90% of the information while discarding noisy, low‑variance dimensions.

### What PCA does in your pipeline

From the logs, for each domain after imputation and standardisation:

| Domain | Original features | PCA components kept | Variance retained |
|--------|-------------------|---------------------|-------------------|
| Kinematic | 8 | 3 | 91% |
| Path | 7 | 2 | 92% |
| Safety | 6 | 5 | 93% |
| Social | 4 | 3 | 96% |

After PCA, clustering operates on these reduced, uncorrelated dimensions. This makes distance calculations more robust and helps clusters separate better.

### The PCA loadings you saw

Files like `kinematic_pca_loadings.csv` tell you how much each original feature contributes to each principal component.  
For example, in the kinematic domain:
- PC1 is dominated by `accel_mean`, `accel_max`, `accel_std`, `jerk_mean`, `jerk_max` (all have high positive loadings). So PC1 is an “agitation” or “acceleration intensity” axis.
- PC2 has high loadings from `speed_mean` (positive) and `stop_ratio` (negative). So PC2 separates “fast continuous walkers” from “frequent stoppers”.
- PC3 is mainly `stop_ratio` with some `speed_std`.

These loadings are the **direction cosines** between original features and the new principal axes. They show that the clusters found later are primarily driven by different combinations of features.

---

## 2. Silhouette Score

### What it measures

The silhouette score quantifies how well each data point lies within its own cluster compared to other clusters. It combines **cohesion** (how close a point is to others in its own cluster) and **separation** (how far it is from points in the nearest other cluster).

For a single point $i$ assigned to cluster $A$:

- $a(i)$ = average distance from $i$ to all other points in $A$ (cohesion).
- $b(i)$ = minimum average distance from $i$ to all points in any other cluster $B \neq A$ (separation).

Then the silhouette for point $i$ is:

$$
s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}}
$$

- If $a(i) \ll b(i)$ (point is much closer to its own cluster than to the nearest other cluster), $s(i) \approx 1$ → **well clustered**.
- If $a(i) \approx b(i)$, $s(i) \approx 0$ → point is on or near the decision boundary.
- If $a(i) \gg b(i)$ (point is closer to another cluster), $s(i)$ becomes negative → **probably assigned to the wrong cluster**.

The **overall silhouette score** for a clustering is the average of $s(i)$ over all points. It ranges from −1 to 1. Higher is better.

### How it's used to choose the number of clusters $k$

The pipeline runs clustering (K‑Means or GMM) for $k = 2, 3, 4, 5$ and computes the silhouette score for each.  
Then it picks the $k$ with the **highest** score.  

From your logs:

- **Kinematic**: k=2 gave silhouette 0.407 (best), k=3 gave 0.402 → chosen 2.
- **Path**: k=2 gave 0.664 (best, quite high) → chosen 2.
- **Safety**: k=3 gave 0.317 (best); k=2 gave 0.308 → chosen 3.
- **Social**: k=3 gave 0.475 (best) → chosen 3.

These scores are moderate (0.3–0.6). Values >0.5 indicate reasonable structure; <0.25 suggest weak or artificial clusters.  
Your Safety domain’s low score (0.317) hints that the three clusters (Isolated, Crowded, Intrusive) are not strongly separated – which makes sense because “intrusive” is just a subset of “crowded” with discrete intrusions, and the boundaries are fuzzy.

### Why silhouette instead of other methods?

The silhouette score:
- Does not require ground truth labels.
- Works with any distance metric.
- Provides a clear per‑point diagnostic (you can find ambiguous points).
- Penalises both too few and too many clusters (if you split a true cluster, $b(i)$ becomes small, lowering the score).

However, it can be computationally expensive for large datasets (O(n²) distances). The pipeline computes it on the PCA‑reduced data, which is much faster.

---

## How it all ties together in your regime discovery

1. **Feature engineering** → 25 raw features across 4 domains.
2. **Preprocessing** → log‑transform skewed features, impute missing values, standardise.
3. **PCA** → reduce each domain to a few uncorrelated components capturing ≥90% variance.
4. **Clustering** → for each domain, run K‑Means/GMM for various $k$, choose the one with highest silhouette score.
5. **Combine** → the cluster labels from the four domains are concatenated (e.g., `0_1_1_0`) to form a regime code.
6. **Filter rare combos** → combinations with fewer than `min_windows_for_regime` windows are merged into `rare_mixed`.

This approach gives you a **principled, data‑driven taxonomy** of walking behaviours without manual labelling. The PCA ensures that noise and collinearity don't dominate the clustering, and the silhouette score objectively determines the number of distinct patterns in each behavioural domain.

If you'd like, I can also explain how GMM differs from K‑Means and why the pipeline offers both options – but that would be another deep dive. Let me know.

---
---
---